The FinBERT pretrained model can be fine-tuned on downstream financial NLP tasks. This notebook illustrates the process of fine-tuning FinBERT using Huggingface's tranformers library. 

In [1]:
# tested in transformers==4.18.0, pytorch==1.7.1 
import torch
import transformers
torch.__version__, transformers.__version__
import numpy as np
import pandas as pd 
from transformers import BertTokenizer, Trainer, BertForSequenceClassification, TrainingArguments
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

/Users/abassjammal/Desktop/VENV/finBERTvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-09-01 17:20:05.181428: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


*Note: the following code is for demonstration purpose. Please use GPU for fast inference on large scale dataset.*

In [2]:
torch.cuda.is_available()

False

### Loading the dataset

In [3]:
# Load the data with 'ISO-8859-1' encoding
df = pd.read_csv('archive/sentiment_cleaned_cnbc_news.csv')

# Display head of cleaned data
print(df.head())


                published_at  \
0  2021-09-29 17:09:39+00:00   
1  2009-04-22 19:49:03+00:00   
2  2018-04-14 14:59:04+00:00   
3  2017-03-07 23:07:14+00:00   
4  2020-12-17 06:08:38+00:00   

                                            keywords  \
0  cnbc, Premium, Articles, Investment strategy, ...   
1  cnbc, Articles, General Motors Co, Business Ne...   
2  cnbc, Articles, George W. Bush, Vladimir Putin...   
3  cnbc, Articles, White House, Oil and Gas, Chev...   
4  cnbc, Articles, World economy, World Markets, ...   

                                       combined_text  token_lengths     label  
0  santolis wednesday market notes could septembe...             68  positive  
1  the us government is considering converting a ...            177  positive  
2  president donald trump declared the us-led int...            141  positive  
3  chevron ceo john watson stated his support for...            117  positive  
4  european stocks closed higher on thursday as t...            134  p

In [ ]:
# # Replace encoding artifacts in the DataFrame
# artifacts = {
#     "Äôs": "'s",
#     "Äî": '"',
#     "¬†": " "
# }

# for artifact, replacement in artifacts.items():
#     df.replace(artifact, replacement, regex=True, inplace=True)

# # Remove rows with 'unknown' in the combined_text column
# df = df[df['combined_text'] != 'unknown']

# # Optionally, save cleaned data
# df.to_csv("cleaned_data.csv", index=False, encoding='utf-8')

# # Display head of cleaned data
# print(df.head())


In [4]:
print(df.columns)

unique_labels_before_mapping = df['label'].unique()
print(unique_labels_before_mapping)

sentiment_mapping = {'neutral': 0, 'positive': 1, 'negative': 2}
df['label'] = df['label'].map(sentiment_mapping)

# Display head of cleaned data
print(df.head())


Index(['published_at', 'keywords', 'combined_text', 'token_lengths', 'label'], dtype='object')
['positive' 'negative' 'neutral']
                published_at  \
0  2021-09-29 17:09:39+00:00   
1  2009-04-22 19:49:03+00:00   
2  2018-04-14 14:59:04+00:00   
3  2017-03-07 23:07:14+00:00   
4  2020-12-17 06:08:38+00:00   

                                            keywords  \
0  cnbc, Premium, Articles, Investment strategy, ...   
1  cnbc, Articles, General Motors Co, Business Ne...   
2  cnbc, Articles, George W. Bush, Vladimir Putin...   
3  cnbc, Articles, White House, Oil and Gas, Chev...   
4  cnbc, Articles, World economy, World Markets, ...   

                                       combined_text  token_lengths  label  
0  santolis wednesday market notes could septembe...             68      1  
1  the us government is considering converting a ...            177      1  
2  president donald trump declared the us-led int...            141      1  
3  chevron ceo john watson stated

### Prepare training/validation/testing

In [5]:
df_train, df_test, = train_test_split(df, stratify=df['label'], test_size=0.1, random_state=42)
df_train, df_val = train_test_split(df_train, stratify=df_train['label'],test_size=0.1, random_state=42)
print(df_train.shape, df_test.shape, df_val.shape)

(474, 5) (59, 5) (53, 5)


### Loading the FinBERT pretrained model
The pretrained FinBERT model path on Huggingface is https://huggingface.co/yiyanghkust/finbert-pretrain

In [6]:
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-pretrain',num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-pretrain')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at yiyanghkust/finbert-pretrain and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Prepare the dataset for fine-tuning

In [7]:
dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)
dataset_test = Dataset.from_pandas(df_test)

dataset_train = dataset_train.map(lambda e: tokenizer(e['combined_text'], truncation=True, padding='max_length', max_length=128), batched=True)
dataset_val = dataset_val.map(lambda e: tokenizer(e['combined_text'], truncation=True, padding='max_length', max_length=128), batched=True)
dataset_test = dataset_test.map(lambda e: tokenizer(e['combined_text'], truncation=True, padding='max_length' , max_length=128), batched=True)

dataset_train.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
dataset_val.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
dataset_test.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])


Map: 100%|██████████| 59/59 [00:00<00:00, 503.86 examples/s]


### Defining the training options

In [8]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy' : accuracy_score(predictions, labels)}

args = TrainingArguments(
        output_dir = 'temp/',
        evaluation_strategy = 'epoch',
        save_strategy = 'epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
)

trainer = Trainer(
        model=model,                         # the instantiated 🤗 Transformers model to be trained
        args=args,                  # training arguments, defined above
        train_dataset=dataset_train,         # training dataset
        eval_dataset=dataset_val,            # evaluation dataset
        compute_metrics=compute_metrics
)

trainer.train()   

/Users/abassjammal/Desktop/VENV/finBERTvenv/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
                                               
 20%|██        | 15/75 [06:46<24:01, 24.02s/it]

{'eval_loss': 0.604331910610199, 'eval_accuracy': 0.7547169811320755, 'eval_runtime': 12.8359, 'eval_samples_per_second': 4.129, 'eval_steps_per_second': 0.156, 'epoch': 1.0}


                                               
 40%|████      | 30/75 [13:26<17:31, 23.36s/it]

{'eval_loss': 0.6010194420814514, 'eval_accuracy': 0.7547169811320755, 'eval_runtime': 12.5334, 'eval_samples_per_second': 4.229, 'eval_steps_per_second': 0.16, 'epoch': 2.0}


                                               
 60%|██████    | 45/75 [19:39<11:18, 22.63s/it]

...

                                               
100%|██████████| 75/75 [32:03<00:00, 22.74s/it]

{'eval_loss': 0.5783764719963074, 'eval_accuracy': 0.7547169811320755, 'eval_runtime': 12.6225, 'eval_samples_per_second': 4.199, 'eval_steps_per_second': 0.158, 'epoch': 5.0}


100%|██████████| 75/75 [32:07<00:00, 25.69s/it]

{'train_runtime': 1927.0067, 'train_samples_per_second': 1.23, 'train_steps_per_second': 0.039, 'train_loss': 0.5912196350097656, 'epoch': 5.0}


TrainOutput(global_step=75, training_loss=0.5912196350097656, metrics={'train_runtime': 1927.0067, 'train_samples_per_second': 1.23, 'train_steps_per_second': 0.039, 'train_loss': 0.5912196350097656, 'epoch': 5.0})

### Evaluate on the testing set

In [9]:
model.eval()
trainer.predict(dataset_test).metrics

100%|██████████| 2/2 [00:06<00:00,  3.08s/it]


{'test_loss': 0.6379687786102295,
 'test_accuracy': 0.7457627118644068,
 'test_runtime': 13.8873,
 'test_samples_per_second': 4.248,
 'test_steps_per_second': 0.144}

### Save the fine-tuned model

In [26]:
trainer.save_model('finbert-cnbc-sentiment/')